In [28]:
from urllib import response

#LOAD ENV VARIABLES
from dotenv import load_dotenv

load_dotenv()

#Create an API client
import os
from google import genai
from google.genai import types

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
model = "gemini-3.1-flash-lite"
#noinspection PyTypeChecker
def add_user_message(messages,content):
    if isinstance(content, str):
        # Handle standard text input from the user
        messages.append(
            types.Content(
                role="user",
                parts=[types.Part.from_text(text=content)]
            )
        )
    elif isinstance(content, list):
        # Handle the list of tool result Parts returned by run_tools()
        messages.append(
            types.Content(
                role="user",
                parts=content
            )
        )

def add_assistant_message(messages,output):
    # Extracts the pre-formatted Content object from Gemini's first candidate
    if output.candidates and output.candidates[0].content:
        messages.append(output.candidates[0].content)

def chat_stream(
        messages,
        system=None,
        temperature=1.0,
        stop_sequences=None,
        tools=None,
        tool_choice=None,
        fine_grained=False,
    ):

    if stop_sequences is None:
        stop_sequences = []
    params = {
        "model":model,
        "contents":messages
    }

    config = types.GenerateContentConfig(temperature=temperature)

    if system:
        config.system_instruction = system

    if stop_sequences:
        config.stop_sequences = stop_sequences

    if tools:
        config.tools = [types.Tool(function_declarations=tools)]
        # Forcing execution mode using the correct SDK configuration layout
        config.tool_config = types.ToolConfig(
            function_calling_config=types.FunctionCallingConfig(
                mode=tool_choice if tool_choice else "AUTO",
                stream_function_call_arguments=True if fine_grained else False
            )
        )

    params["config"] = config

    message = client.models.generate_content_stream(**params)
    return message

def text_from_message(message):
    # Returns the combined text, or an empty string if no text exists
    return message.text if message.text else ""

In [29]:
# Tool definition
save_article_schema = {
        "name": "save_article",
        "description": "Saves a scholarly journal article",
        "parameters": {
            "type": "OBJECT",
            "properties": {
                "abstract": {
                    "type": "STRING",
                    "description": "Abstract of the article. One short sentence max",
                },
                "meta": {
                    "type": "OBJECT",
                    "properties": {
                        "word_count": {
                            "type": "integer",
                            "description": "Word count",
                        },
                        "review": {
                            "type": "STRING",
                            "description": "Eight sentence review of the paper",
                        },
                    },
                    "required": ["word_count", "review"],
                },
            },
            "required": ["abstract", "meta"],
        },
    }

save_short_article_schema = {
        "name": "save_article",
        "description": "Saves a scholarly journal article",
        "parameters": {
            "type": "OBJECT",
            "properties": {
                "abstract": {
                    "type": "STRING",
                    "description": "Abstract of the article. One short sentence max",
                },
                "meta": {
                    "type": "OBJECT",
                    "properties": {
                        "word_count": {
                            "type": "integer",
                            "description": "Word count",
                        },
                        "review": {
                            "type": "STRING",
                            "description": "Review of paper. One short sentence max",
                        },
                    },
                    "required": ["word_count", "review"],
                },
            },
            "required": ["abstract", "meta"],
        },
    }


def save_article(**kwargs):
    return "Article saved!"


In [30]:
import json


def run_tool(tool_name, tool_input):
    if tool_name == "save_article":
        return save_article(**tool_input)
    return None


def run_tools(message):
    tool_result_parts = []

    # 1. Gemini stores requested tools directly in response.function_calls
    if not message.function_calls:
        return tool_result_parts

    for tool_request in message.function_calls:
        try:
            # tool_request.name is the function name string
            # tool_request.args is a Python dict containing the arguments
            tool_output = run_tool(tool_request.name, tool_request.args)

            # Successful response payload structure
            response_payload = {"result": tool_output}

        except Exception as e:
            # Error response payload structure
            response_payload = {"error": str(e)}

        # 2. Package the result into Gemini's official Part format
        tool_result_part = types.Part.from_function_response(
            name=tool_request.name,
            response=response_payload
        )

        tool_result_parts.append(tool_result_part)

    return tool_result_parts

In [31]:
# Run conversation
def run_conversation(messages, tools=[], tool_choice=None,fine_grained=False):
    while True:
        # Start the stream
        stream = chat_stream(
            messages,
            tools=tools,
            tool_choice=tool_choice,
            fine_grained=["fine-grained-tool-streaming-2025-05-14"] if fine_grained else []
        )

        accumulated_parts = []
        has_function_calls = False

        # Iterate through chunks as they arrive
        for chunk in stream:
            if chunk.candidates and chunk.candidates[0].content and chunk.candidates[0].content.parts:
                for part in chunk.candidates[0].content.parts:

                    # 1. Access part.text directly (This silences the warning message)
                    if part.text:
                        print(part.text, end="", flush=True)

                    # Track if a tool execution was requested in this chunk
                    if part.function_call:
                        has_function_calls = True

                    accumulated_parts.append(part)

        print() # Print a line break when this turn's streaming ends

        # Guard clause: If the model returned absolutely nothing, break out
        if not accumulated_parts:
            break

        # 2. Build a local helper class to stitch the parts into a complete response object
        class AccumulatedResponse:
            def __init__(self, parts):
                from google.genai import types
                self.candidates = [
                    types.Candidate(
                        content=types.Content(role="model", parts=parts)
                    )
                ]
            @property
            def function_calls(self):
                return [p.function_call for p in self.candidates[0].content.parts if p.function_call]

        # Combine all stream fragments into a unified response
        full_response = AccumulatedResponse(accumulated_parts)

        # 3. Save the full, unified response turn to your history
        add_assistant_message(messages, full_response)

        # 4. If no tools were called across the entire stream, we are safely finished!
        if not has_function_calls:
            break

        # 5. Execute tools using the complete collected data (Fixes the 2-minute loop hang)
        tool_results = run_tools(full_response)
        add_user_message(messages, tool_results)
    return messages

In [32]:
messages = []

add_user_message(
    messages,
    "Create and save a fake computer science article",
)

run_conversation(
    messages,
    tools=[save_article_schema],
    fine_grained=True
)

ValueError: stream_function_call_arguments parameter is only supported in Gemini Enterprise Agent Platform mode, not in Gemini Developer API mode.